In [45]:
import os
from langchain.chat_models import init_chat_model
from langchain.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, PromptTemplate

In [7]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00


In [9]:
pip install langchain

In [46]:
GROQ_APT_KEY ='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'

In [42]:
pip install -U langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 5.6 MB/s eta 0:00:00


In [47]:
#Initialize the chat model
model = init_chat_model("gemma2-9b-it",model_provider="groq",api_key=GROQ_APT_KEY)

In [20]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 7.3 MB/s eta 0:00:00


In [23]:
from google.colab import files
uploaded = files.upload()

Saving Statistics Notes  (1).pdf to Statistics Notes  (1).pdf


In [24]:
#Load PDF document
loader = PyPDFLoader(r'/content/Statistics Notes  (1).pdf')
docs = loader.load()

In [48]:
#Split documnets into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [33]:
pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 68.9 MB/s eta 0:00:00


In [49]:
#Initialize embeddings and vector store
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(splits, embeddings_model)
retriever = vectorstore.as_retriever()

In [55]:
# Define prompt template
prompt = ChatPromptTemplate(
    imput_variables=["context", "question"],
    messages=[
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=["context", "question"],
                template=(
                    "You are an assistant for question-answering tasks."
                    "Use the following retrieved context to answer the question."
                    "If you don't know the answer, just say that you don't know."
                    "Use three sentences maximum and keep the answer concise."
                    "Question: {question}\n"
                    "Context: {context}\n"
                    "Answer:"
                )
            )

        )
    ]

)

In [56]:
#Function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [57]:
#Defin the RAG chain
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [60]:
#Invoke the RAG chain
response = rag_chain.invoke("What is Standard Deviation?")
print(response)

Standard deviation is a measure of how spread out data points are from the mean.  It is calculated as the square root of the variance. A larger standard deviation indicates greater data variability.  

